In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from scipy.signal import butter, filtfilt
%matplotlib inline

Комірка 2: Основна логіка та генерація даних
Тут ми створюємо клас для збереження стану шуму. Це необхідно для виконання вимоги: якщо параметри гармоніки змінюються, а шуму — ні, то шум не повинен генеруватися наново. Також тут реалізована запитувана функція harmonic_with_noise.

In [2]:
class HarmonicData:
    def __init__(self):
        self.t = np.linspace(0, 10, 1000)
        self.current_noise = None
        self.last_noise_mean = None
        self.last_noise_cov = None

    def get_noise(self, mean, covariance):
        if (self.current_noise is None or 
            mean != self.last_noise_mean or 
            covariance != self.last_noise_cov):
            
            std_dev = np.sqrt(max(0, covariance)) 
            self.current_noise = np.random.normal(mean, std_dev, len(self.t))
            
            self.last_noise_mean = mean
            self.last_noise_cov = covariance
            
        return self.current_noise

def harmonic_with_noise(t, amplitude, frequency, phase, noise_mean, noise_covariance, show_noise, data_manager):
    clean_harmonic = amplitude * np.sin(frequency * t + phase)
    
    noise = data_manager.get_noise(noise_mean, noise_covariance)
    
    noisy_harmonic = clean_harmonic + noise
    
    if not show_noise:
        return clean_harmonic, clean_harmonic
    
    return clean_harmonic, noisy_harmonic

def apply_filter(data, cutoff_freq, fs=100, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff_freq / nyq
    if normal_cutoff >= 1.0 or normal_cutoff <= 0:
        return data
    b, a = butter(order, normal_cutoff, btype='low', analog=False)
    filtered_data = filtfilt(b, a, data)
    return filtered_data

Комірка 3: Інтерфейс та візуалізація
Ця комірка створює слайдери, чекбокси, кнопку та пов'язує їх з функцією малювання графіка. Програма одразу оновлюватиме графік при зміні параметрів.

In [ ]:
data_manager = HarmonicData()

INITIAL_PARAMS = {
    'amplitude': 1.0,
    'frequency': 1.0,
    'phase': 0.0,
    'noise_mean': 0.0,
    'noise_cov': 0.1,
    'cutoff_freq': 5.0,
    'show_noise': True
}

style = {'description_width': 'initial'}
amp_slider = widgets.FloatSlider(value=INITIAL_PARAMS['amplitude'], min=0.1, max=5.0, step=0.1, description='Amplitude:', style=style)
freq_slider = widgets.FloatSlider(value=INITIAL_PARAMS['frequency'], min=0.1, max=10.0, step=0.1, description='Frequency:', style=style)
phase_slider = widgets.FloatSlider(value=INITIAL_PARAMS['phase'], min=0.0, max=2*np.pi, step=0.1, description='Phase:', style=style)
noise_mean_slider = widgets.FloatSlider(value=INITIAL_PARAMS['noise_mean'], min=-2.0, max=2.0, step=0.1, description='Noise Mean:', style=style)
noise_cov_slider = widgets.FloatSlider(value=INITIAL_PARAMS['noise_cov'], min=0.0, max=2.0, step=0.05, description='Noise Covariance:', style=style)
cutoff_slider = widgets.FloatSlider(value=INITIAL_PARAMS['cutoff_freq'], min=0.1, max=49.9, step=0.5, description='Cutoff Frequency:', style=style)
show_noise_checkbox = widgets.Checkbox(value=INITIAL_PARAMS['show_noise'], description='Show Noise')
reset_button = widgets.Button(description='Reset', button_style='info')

def update_plot(amplitude, frequency, phase, noise_mean, noise_cov, cutoff_freq, show_noise):
    plt.figure(figsize=(10, 6))
    
    clean, noisy = harmonic_with_noise(
        data_manager.t, amplitude, frequency, phase, 
        noise_mean, noise_cov, show_noise, data_manager
    )
    
    filtered = apply_filter(noisy, cutoff_freq)
    
    if show_noise:
        plt.plot(data_manager.t, noisy, label='Зашумлена гармоніка', color='orange', alpha=0.7)
    
    plt.plot(data_manager.t, clean, label='Чиста гармоніка', color='blue', linewidth=2)
    plt.plot(data_manager.t, filtered, label='Відфільтрована гармоніка', color='red', linestyle='--', linewidth=2)
    
    plt.title('Візуалізація гармоніки з шумом та фільтрацією')
    plt.xlabel('Час (t)')
    plt.ylabel('Амплітуда (y)')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.show()

def reset_values(b):
    amp_slider.value = INITIAL_PARAMS['amplitude']
    freq_slider.value = INITIAL_PARAMS['frequency']
    phase_slider.value = INITIAL_PARAMS['phase']
    noise_mean_slider.value = INITIAL_PARAMS['noise_mean']
    noise_cov_slider.value = INITIAL_PARAMS['noise_cov']
    cutoff_slider.value = INITIAL_PARAMS['cutoff_freq']
    show_noise_checkbox.value = INITIAL_PARAMS['show_noise']

reset_button.on_click(reset_values)

out = widgets.interactive_output(update_plot, {
    'amplitude': amp_slider,
    'frequency': freq_slider,
    'phase': phase_slider,
    'noise_mean': noise_mean_slider,
    'noise_cov': noise_cov_slider,
    'cutoff_freq': cutoff_slider,
    'show_noise': show_noise_checkbox
})

left_box = widgets.VBox([amp_slider, freq_slider, phase_slider])
right_box = widgets.VBox([noise_mean_slider, noise_cov_slider, cutoff_slider])
controls = widgets.HBox([left_box, right_box])
bottom_controls = widgets.HBox([show_noise_checkbox, reset_button])
ui = widgets.VBox([controls, bottom_controls])

display(ui, out)

Output()